In [6]:
!ls /kaggle/input/

datasets


In [7]:
!ls -R /kaggle/input | head -40

/kaggle/input:
datasets

/kaggle/input/datasets:
akshatyuvan

/kaggle/input/datasets/akshatyuvan:
fakemusiccaps-raw
fmc-manifest
musiccaps-real-16k

/kaggle/input/datasets/akshatyuvan/fakemusiccaps-raw:
audioldm2
MusicGen_medium
musicldm
mustango
stable_audio_open

/kaggle/input/datasets/akshatyuvan/fakemusiccaps-raw/audioldm2:
00M9FhCet6s.wav
01hjVJN9xCg.wav
01PzcPKT3_E.wav
0298WjE3_tk.wav
_0-2meOf9qY.wav
02Qntw26enM.wav
03frQGyrgQ4.wav
03z0rpIkm5g.wav
04NjXsLyCl4.wav
05JAmKFVy44.wav
05OJDYeHLMc.wav
06Brdf83RZE.wav
06IU9WsEp3s.wav
07-3grCQ8EU.wav
07FxCXxknY4.wav
07mS0mSTDjY.wav
0-7PyzhzuYQ.wav
07-vpXo91XM.wav
07xGXxIHOL4.wav
08mf5GxT820.wav
08u-jdwjM74.wav
ls: write error: Broken pipe


In [8]:
import os
R = '/kaggle/input/datasets/akshatyuvan/musiccaps-real-16k'
print(os.listdir(R)[:5])
print('entries:', len(os.listdir(R)))

['-R0267o4lLk.wav', 'n7yLkcSfiuM.wav', '8fInAz_GICs.wav', '8kH-dzSBthI.wav', '6TSluKI0X54.wav']
entries: 5352


In [10]:
print(os.listdir('/kaggle/input/datasets/akshatyuvan/fmc-manifest'))

['manifest.csv']


In [11]:
import os, pandas as pd

REAL = '/kaggle/input/datasets/akshatyuvan/musiccaps-real-16k'
SYN  = '/kaggle/input/datasets/akshatyuvan/fakemusiccaps-raw'

man = pd.read_csv('/kaggle/input/datasets/akshatyuvan/fmc-manifest/manifest.csv')

# Manifest paths were written against last session's working dirs.
# Repoint both classes at this session's mounted datasets.
man['path'] = (man.path
    .str.replace('/kaggle/working/real', REAL, regex=False)
    .str.replace('/kaggle/input/datasets/akshatyuvan/fakemusiccaps-raw', SYN, regex=False))

print(man.shape, man.split.value_counts().to_dict())
print('real ok :', os.path.exists(man[man.label==0].path.iloc[0]))
print('synth ok:', os.path.exists(man[man.label==1].path.iloc[0]))

(32112, 5) {'train': 22476, 'test': 4824, 'val': 4812}
real ok : True
synth ok: True


In [12]:
SEED = 42
# Manifest is 1:5 real:synthetic. For a preliminary binary baseline we take a
# balanced subsample within each split, drawing synthetic clips evenly across
# the five generators so no single generator dominates the fake class.
parts = []
for sp in ['train','val','test']:
    d = man[man.split == sp]
    real = d[d.label == 0]
    n_per_gen = len(real) // 5
    fake = (d[d.label == 1]
              .groupby('generator', group_keys=False)
              .apply(lambda g: g.sample(n_per_gen, random_state=SEED)))
    parts += [real, fake]

sub = pd.concat(parts).reset_index(drop=True)
print(sub.groupby(['split','label']).size())

split  label
test   0         804
       1         800
train  0        3746
       1        3745
val    0         802
       1         800
dtype: int64


/tmp/ipykernel_49/2088331323.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n_per_gen, random_state=SEED)))
/tmp/ipykernel_49/2088331323.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n_per_gen, random_state=SEED)))
/tmp/ipykernel_49/2088331323.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and

In [13]:
import numpy as np, librosa, soundfile as sf
from joblib import Parallel, delayed

N_SAMPLES = 160000   # exactly 10.0 s at 16 kHz. Real clips are 10.01 s, so
                     # cropping removes clip length as a possible shortcut.

def feats(path):
    y, sr = sf.read(path, dtype='float32')
    y = y[:N_SAMPLES] if len(y) >= N_SAMPLES else np.pad(y, (0, N_SAMPLES - len(y)))
    S = np.abs(librosa.stft(y, n_fft=2048, hop_length=512))
    blocks = [
        librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20),
        librosa.feature.chroma_stft(S=S, sr=sr),
        librosa.feature.spectral_contrast(S=S, sr=sr),
        librosa.feature.spectral_centroid(S=S, sr=sr),
        librosa.feature.spectral_rolloff(S=S, sr=sr),
        librosa.feature.spectral_bandwidth(S=S, sr=sr),
        librosa.feature.zero_crossing_rate(y),
    ]
    # mean and std over time for each descriptor -> one fixed-length vector per clip
    return np.concatenate([np.concatenate([b.mean(1), b.std(1)]) for b in blocks])

X = np.array(Parallel(n_jobs=4, verbose=5)(delayed(feats)(p) for p in sub.path))
y = sub.label.values
print(X.shape, y.shape, 'NaNs:', np.isnan(X).sum())

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  10 tasks      | elapsed:   19.8s
[Parallel(n_jobs=4)]: Done  84 tasks      | elapsed:   21.1s
[Parallel(n_jobs=4)]: Done 264 tasks      | elapsed:   24.5s
[Parallel(n_jobs=4)]: Done 516 tasks      | elapsed:   29.2s
[Parallel(n_jobs=4)]: Done 840 tasks      | elapsed:   35.1s
[Parallel(n_jobs=4)]: Done 1236 tasks      | elapsed:   42.3s
[Parallel(n_jobs=4)]: Done 1704 tasks      | elapsed:   51.1s
[Parallel(n_jobs=4)]: Done 2244 tasks      | elapsed:  1.0min
[Parallel(n_jobs=4)]: Done 2856 tasks      | elapsed:  1.2min
[Parallel(n_jobs=4)]: Done 3540 tasks      | elapsed:  1.4min
[Parallel(n_jobs=4)]: Done 4296 tasks      | elapsed:  1.7min
[Parallel(n_jobs=4)]: Done 5124 tasks      | elapsed:  1.9min
[Parallel(n_jobs=4)]: Done 6024 tasks      | elapsed:  2.2min
[Parallel(n_jobs=4)]: Done 6996 tasks      | elapsed:  2.5min
[Parallel(n_jobs=4)]: Done 8040 tasks      | elapsed:  2.8min


(10697, 86) (10697,) NaNs: 0


[Parallel(n_jobs=4)]: Done 10690 out of 10697 | elapsed:  3.7min remaining:    0.1s
[Parallel(n_jobs=4)]: Done 10697 out of 10697 | elapsed:  3.7min finished


In [14]:
import numpy as np, os, json
os.makedirs('/kaggle/working/m1_features', exist_ok=True)
np.save('/kaggle/working/m1_features/X.npy', X)
np.save('/kaggle/working/m1_features/y.npy', y)
sub[['path','ytid','generator','label','split']].to_csv(
    '/kaggle/working/m1_features/subset.csv', index=False)
json.dump({"title":"fmc-m1-features","id":"akshatyuvan/fmc-m1-features",
           "licenses":[{"name":"CC0-1.0"}]},
          open('/kaggle/working/m1_features/dataset-metadata.json','w'))
!ls -lh /kaggle/working/m1_features

total 8.2M
-rw-r--r-- 1 root root  100 Oct  3 21:04 dataset-metadata.json
-rw-r--r-- 1 root root 1.1M Oct  3 21:04 subset.csv
-rw-r--r-- 1 root root 7.1M Oct  3 21:04 X.npy
-rw-r--r-- 1 root root  84K Oct  3 21:04 y.npy


In [16]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['KAGGLE_API_TOKEN'] = UserSecretsClient().get_secret("kaggle_token")
!kaggle datasets create -p /kaggle/working/m1_features

Starting upload for file X.npy
100%|██████████████████████████████████████| 7.02M/7.02M [00:02<00:00, 2.75MB/s]
Upload successful: X.npy (7MB)
Starting upload for file subset.csv
100%|███████████████████████████████████████| 1.06M/1.06M [00:01<00:00, 689kB/s]
Upload successful: subset.csv (1MB)
Starting upload for file y.npy
100%|███████████████████████████████████████| 83.7k/83.7k [00:00<00:00, 103kB/s]
Upload successful: y.npy (84KB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/akshatyuvan/fmc-m1-features


In [17]:
import numpy as np, torch, torch.nn as nn, time
from sklearn.preprocessing import StandardScaler

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'

idx = {s: (sub.split == s).values for s in ['train','val','test']}

# Scaler is fit on train only. Fitting on all data would leak test statistics.
sc = StandardScaler().fit(X[idx['train']])
Xs = {s: torch.tensor(sc.transform(X[idx[s]]), dtype=torch.float32, device=dev)
      for s in idx}
ys = {s: torch.tensor(y[idx[s]], dtype=torch.float32, device=dev) for s in idx}

model = nn.Sequential(
    nn.Linear(X.shape[1], 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(256, 128),        nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(128, 1)
).to(dev)

opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
lossf = nn.BCEWithLogitsLoss()

best_val, best_state, patience, bad = 0.0, None, 15, 0
t0 = time.time()
for ep in range(200):
    model.train()
    perm = torch.randperm(len(Xs['train']), device=dev)
    for i in range(0, len(perm), 128):
        b = perm[i:i+128]
        opt.zero_grad()
        loss = lossf(model(Xs['train'][b]).squeeze(1), ys['train'][b])
        loss.backward(); opt.step()

    model.eval()
    with torch.no_grad():
        va = ((model(Xs['val']).squeeze(1) > 0).float() == ys['val']).float().mean().item()
    if va > best_val:
        best_val, bad = va, 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        bad += 1
        if bad >= patience:
            print('early stop at epoch', ep); break

model.load_state_dict(best_state)
train_time = time.time() - t0
print(f'best val acc: {best_val:.4f} | train time: {train_time:.1f}s')

early stop at epoch 43
best val acc: 0.9157 | train time: 7.4s


In [19]:
import numpy as np
from sklearn.metrics import (balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)

model.eval()
with torch.no_grad():
    logits = model(Xs['test']).squeeze(1)
    score  = torch.sigmoid(logits).cpu().numpy()
pred = (score > 0.5).astype(int)
true = y[idx['test']]

fpr, tpr, _ = roc_curve(true, score)
fnr = 1 - tpr
eer = float(fpr[np.nanargmin(np.abs(fnr - fpr))])   # where FAR and FRR cross

m1 = {
    "balanced_accuracy": balanced_accuracy_score(true, pred),
    "precision": precision_score(true, pred),
    "recall":    recall_score(true, pred),
    "f1":        f1_score(true, pred),
    "auc":       roc_auc_score(true, score),
    "eer":       eer,
}
for k, v in m1.items():
    print(f"{k:20s} {v:.4f}")
m1_time = train_time
print(f"\ntrain time {m1_time:.1f}s | seed {SEED} | device {dev}")

balanced_accuracy    0.8972
precision            0.8821
recall               0.9163
f1                   0.8988
auc                  0.9692
eer                  0.0983

train time 7.4s | seed 42 | device cpu


In [20]:
import json
json.dump({"model":"M1_MLP","seed":SEED,"device":dev,"train_time_s":7.4,
           "best_val_acc":0.9157,"early_stop_epoch":43,
           "balanced_accuracy":0.8972,"precision":0.8821,"recall":0.9163,
           "f1":0.8988,"auc":0.9692,"eer":0.0983},
          open('/kaggle/working/m1_features/m1_results.json','w'), indent=2)
print(open('/kaggle/working/m1_features/m1_results.json').read())

{
  "model": "M1_MLP",
  "seed": 42,
  "device": "cpu",
  "train_time_s": 7.4,
  "best_val_acc": 0.9157,
  "early_stop_epoch": 43,
  "balanced_accuracy": 0.8972,
  "precision": 0.8821,
  "recall": 0.9163,
  "f1": 0.8988,
  "auc": 0.9692,
  "eer": 0.0983
}


In [21]:
import os
from kaggle_secrets import UserSecretsClient
os.environ['KAGGLE_API_TOKEN'] = UserSecretsClient().get_secret("kaggle_token")
!kaggle datasets create -p /kaggle/working/m1_features

Dataset creation error: The requested title "fmc-m1-features" is already in use by a dataset. Please choose another title.


In [22]:
!kaggle datasets list -m

ref                             title                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------  ------------------  ----------  --------------------------  -------------  ---------  ---------------  
akshatyuvan/tweets              tweets                 6791606  2026-04-14 17:51:09.203000              0          0  0.11764706       
akshatyuvan/fakemusiccaps-raw   fakemusiccaps-raw            0  2026-10-03 13:54:08.600000              0          0  0.25             
akshatyuvan/fmc-manifest        fmc-manifest            401901  2026-10-03 16:02:51.093000              0          0  0.23529412       
akshatyuvan/musiccaps-real-16k  musiccaps-real-16k           0  2026-10-03 16:01:02.200000              0          0  0.25             
akshatyuvan/fmc-m1-features     fmc-m1-features        5121139  2026-10-03 21:10:50.937000              0          0  0.23529412       


In [23]:
!kaggle datasets version -p /kaggle/working/m1_features -m "add m1 results"

Starting upload for file m1_results.json
100%|████████████████████████████████████████████| 255/255 [00:00<00:00, 311B/s]
Upload successful: m1_results.json (255B)
Starting upload for file X.npy
100%|██████████████████████████████████████| 7.02M/7.02M [00:02<00:00, 3.09MB/s]
Upload successful: X.npy (7MB)
Starting upload for file subset.csv
100%|███████████████████████████████████████| 1.06M/1.06M [00:02<00:00, 466kB/s]
Upload successful: subset.csv (1MB)
Starting upload for file y.npy
100%|██████████████████████████████████████| 83.7k/83.7k [00:01<00:00, 61.9kB/s]
Upload successful: y.npy (84KB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/akshatyuvan/fmc-m1-features
